# Equilibrium lattice parameter and bulk modulus of MgO

*Based on Problems 5 and 6 of the [ICTP-MARVEL College 2026](https://github.com/marvel-nccr/ictp-marvel-college-2026) day-01 exercise, originally authored by Edward Linscott.*

In the [convergence notebook](qe_convergence_tests.ipynb) you determined converged values of `ecutwfc` and the **k**-point grid for MgO. Here you will use those values to:

1. **Problem 5** — compute the total energy as a function of lattice parameter and fit a parabola to locate the equilibrium value $a_0$.
2. **Problem 6** — extract the bulk modulus $B$ from the curvature of the $E(V)$ parabola, and optionally refine it with a third-order Birch–Murnaghan equation of state.

The bulk modulus is defined as

$$B = -V_0 \left.\frac{\partial P}{\partial V}\right|_{V=V_0},$$

where $P$ is the pressure, $V$ is the cell volume, and $V_0$ is the equilibrium volume. It measures the resistance of a material to uniform compression.

## Problem 5: The equilibrium lattice parameter

To locate the energy minimum we sweep the lattice parameter $a$ around the value used in the convergence tests ($a = 4.21$ Å, `celldm(1)` $\approx 7.955$ bohr) and compute the total energy at each point.

We sample the interval $[-7\%,\, +9\%]$ relative to $a = 4.21$ Å, i.e.

$$a_i = 4.21 \times (1 + s_i) \quad \text{Å}, \qquad s_i \in [-0.07,\, +0.09],$$

using 11 evenly spaced values (roughly 1.5 % steps). The asymmetric window is intentional: the $E(V)$ curve is steeper on the compressed side, so we extend further on the expanded side to capture the flat region near the minimum.

For each lattice parameter we run a single SCF calculation with the converged `ecutwfc` and **k**-point grid from the convergence notebook. Because we are sweeping `celldm(1)` we reuse `build_mgo_input` directly — only the `atoms` object (and hence `celldm(1)`) changes between cases.

In [ ]:
from pathlib import Path
import glob, shutil
import numpy as np
from ase.build import bulk

from pw_input import (
    ControlNamelist, SystemNamelist, ElectronsNamelist,
    AtomicSpeciesCard, AtomicPositionsCard, KPointsAutoCard, PWInput,
)
from convergence_runner import QERunner, RY_TO_EV, BOHR_TO_ANG

RUN_ROOT  = Path('.').resolve()
PSEUDO_DIR = RUN_ROOT / 'pseudo'
EOS_DIR   = RUN_ROOT / 'out' / 'eos'
EOS_DIR.mkdir(parents=True, exist_ok=True)

_pw_candidates = sorted(glob.glob('/home/pietro/repositories/q-e/build_*/bin/pw.x'))
PW_CMD = [_pw_candidates[0]] if _pw_candidates else (
    [shutil.which('pw.x')] if shutil.which('pw.x') else None
)
if PW_CMD is None:
    raise RuntimeError('pw.x not found.')

PSEUDOS = {
    'Mg': 'Mg.upf',
    'O':  'O.upf',
}

In [ ]:
def build_mgo_input(atoms, ecutwfc, nk, prefix):
    control   = ControlNamelist(calculation='scf', prefix=prefix,
                                pseudo_dir=str(PSEUDO_DIR), outdir=str(EOS_DIR),
                                tprnfor=True, tstress=True)
    system    = SystemNamelist.from_atoms(atoms, ibrav=2, ecutwfc=ecutwfc)
    electrons = ElectronsNamelist()
    species   = AtomicSpeciesCard.from_atoms(atoms, PSEUDOS)
    positions = AtomicPositionsCard.from_atoms(atoms, units='crystal')
    kpoints   = KPointsAutoCard(2, nk=nk)
    return PWInput(control, system, electrons, species, positions, kpoints)

> [!NOTE]
> **Pulay stress — choose your cutoff before running.** `pw.x` evaluates the stress tensor as the analytic derivative of the DFT energy with respect to cell strain *at fixed plane-wave coefficients*. This omits the contribution from the basis-set change under strain — the **Pulay stress** — a systematic bias that shifts the computed pressure toward more compressive values and converges slowly with `ecutwfc`, far more slowly than the total energy (which is protected by the variational principle).
>
> **Quick diagnostic.** At equilibrium the pressure must be zero and the energy must be at its minimum — both at the *same* lattice parameter. Inspect the $P_{\rm QE}$ column in the table alongside the energies: if the sign change of $P_{\rm QE}$ falls at a larger $a$ than the energy minimum, the Pulay stress is too large and the cutoff needs to be increased.
>
> Start with `ECUTWFC_CONV = 60` Ry. If the check fails, increase in steps (65, 70, 75, 80 Ry …) and re-run until the sign change of $P_{\rm QE}$ coincides with the energy minimum. Use that value for the rest of the exercise.

In [ ]:
# Fill in from your convergence notebook (Problems 1–4)
ECUTWFC_CONV = 60   # Ry
NK_CONV      = 4    # nk×nk×nk Monkhorst–Pack grid

A_REF        = 4.21                          # Å — experimental MgO lattice parameter
SCALE_VALUES = np.linspace(-0.07, 0.09, 11)  # 11 points: -7% … +9%
FORCE_RERUN  = False

runner = QERunner(PW_CMD)

eos_cases = [
    (f'eos_{i:02d}', build_mgo_input(
        bulk('MgO', 'rocksalt', a=A_REF * (1 + s)),
        ECUTWFC_CONV, NK_CONV,
        prefix=f'mgo_eos_{i:02d}',
    ))
    for i, s in enumerate(SCALE_VALUES)
]

eos_results = runner.run_sweep(eos_cases, EOS_DIR, force_rerun=FORCE_RERUN,
                               collect_pressure=True)

a_values   = A_REF * (1 + SCALE_VALUES)
V_values   = np.array([bulk('MgO', 'rocksalt', a=a).get_volume() for a in a_values])  # Å³
E_values   = np.array([r['energy_ry'] * RY_TO_EV for r in eos_results])               # eV/cell
P_qe_gpa   = np.array([r['pressure_kbar'] * 0.1   for r in eos_results])              # GPa

print(f"{'a (Å)':>8}  {'V (Å³)':>8}  {'E (eV)':>14}  {'P_QE (GPa)':>11}")
for a, v, e, p in zip(a_values, V_values, E_values, P_qe_gpa):
    print(f'{a:8.4f}  {v:8.4f}  {e:14.6f}  {p:11.2f}')

### Parabolic fit — equilibrium lattice parameter $a_0$

We fit a second-degree polynomial to $E(a)$:

$$E(a) \approx p_2\, a^2 + p_1\, a + p_0, \qquad a_0 = -\frac{p_1}{2\,p_2}$$

<details>
<summary>Implementation: <code>fit_parabola</code> (<code>eos_tools.py</code>)</summary>

```python
def fit_parabola(x, y):
    coeffs = np.polyfit(x, y, 2)           # [c2, c1, c0]
    poly   = np.poly1d(coeffs)
    x0     = -coeffs[1] / (2 * coeffs[0])  # vertex
    return poly, x0, poly(x0)
```

</details>

In [ ]:
import matplotlib.pyplot as plt
from eos_tools import fit_parabola

poly_a, a0, _ = fit_parabola(a_values, E_values)

a_fine = np.linspace(a_values[0], a_values[-1], 300)
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(a_values, E_values, zorder=3, label='DFT points')
ax.plot(a_fine, poly_a(a_fine), label='Parabolic fit')
ax.axvline(a0, ls='--', color='gray', label=f'$a_0 = {a0:.4f}$ Å')
ax.set_xlabel('Lattice parameter $a$ (Å)')
ax.set_ylabel('Total energy (eV/cell)')
ax.set_title('MgO — $E$ vs lattice parameter')
ax.legend()
fig.tight_layout()
plt.show()

print(f'Equilibrium lattice parameter:  a₀ = {a0:.4f} Å  (experiment: 4.21 Å)')

## Problem 6: Bulk modulus

For a quadratic $E(V) = c_2 V^2 + c_1 V + c_0$, the equilibrium volume is $V_0 = -c_1/(2c_2)$ and

$$B = V_0 \left.\frac{d^2 E}{d V^2}\right|_{V_0} = 2\,c_2\,V_0$$

with $E$ in eV and $V$ in Å³ (multiply by 160.218 to get GPa). For a more accurate result we use the third-order **Birch–Murnaghan** EOS:

$$E(V) = E_0 + \frac{9 V_0 B_0}{16}\left[
B_0'(\eta-1)^3 + (\eta-1)^2(6 - 4\eta)
\right], \quad \eta = \left(\frac{V_0}{V}\right)^{2/3}$$

> **Cell volume:** `bulk('MgO','rocksalt',a)` returns the fcc primitive cell (2 atoms),
> so $V = a^3/4$ and $a_0 = (4V_0)^{1/3}$.

<details>
<summary>Implementation: <code>birch_murnaghan</code> and <code>fit_bm_eos</code> (<code>eos_tools.py</code>)</summary>

```python
def birch_murnaghan(V, E0, V0, B0, B0p):
    eta = (V0 / V) ** (2 / 3)
    return E0 + (9 * V0 * B0 / 16) * (
        B0p * (eta - 1) ** 3 + (eta - 1) ** 2 * (6 - 4 * eta)
    )

def fit_bm_eos(V, E):
    poly_v, V0_par, E0_par = fit_parabola(V, E)
    B0_par = 2 * poly_v.coeffs[0] * V0_par   # eV/Å³ — initial guess from parabola
    popt, _ = curve_fit(birch_murnaghan, V, E, p0=[E0_par, V0_par, B0_par, 4.0])
    E0, V0, B0, B0p = popt
    return {'E0': E0, 'V0': V0, 'B0_gpa': B0 * EV_ANG3_TO_GPA, 'B0p': B0p, 'popt': popt}
```

</details>

In [ ]:
from eos_tools import fit_parabola, fit_bm_eos, birch_murnaghan, EV_ANG3_TO_GPA

poly_v, V0_par, _ = fit_parabola(V_values, E_values)
B_par  = 2 * poly_v.coeffs[0] * V0_par * EV_ANG3_TO_GPA
a0_par = (4 * V0_par) ** (1/3)

bm    = fit_bm_eos(V_values, E_values)
a0_bm = (4 * bm['V0']) ** (1/3)

print(f"{'Method':<20}  {'a₀ (Å)':>9}  {'B₀ (GPa)':>10}  {'B₀′':>5}")
print(f"{'Parabola E(V)':<20}  {a0_par:>9.4f}  {B_par:>10.1f}")
print(f"{'Birch–Murnaghan':<20}  {a0_bm:>9.4f}  {bm['B0_gpa']:>10.1f}  {bm['B0p']:>5.2f}")
print(f"{'Experiment':<20}  {'4.21':>9}  {'~160':>10}  {'~4':>5}")

V_fine = np.linspace(V_values[0], V_values[-1], 300)
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(V_values, E_values, zorder=3, label='DFT points')
ax.plot(V_fine, poly_v(V_fine), ls='--', label='Parabolic fit')
ax.plot(V_fine, birch_murnaghan(V_fine, *bm['popt']), label='Birch–Murnaghan')
ax.axvline(bm['V0'], ls=':', color='gray', label=f"$V_0 = {bm['V0']:.3f}$ Å³")
ax.set_xlabel('Volume (Å³/cell)')
ax.set_ylabel('Total energy (eV/cell)')
ax.set_title('MgO — equation of state')
ax.legend()
fig.tight_layout()
plt.show()

## Stress check: fitted vs directly computed pressure

The pressure from the EOS fit ($P = -dE/dV$) and the hydrostatic pressure $P = -\frac{1}{3}\mathrm{Tr}(\boldsymbol{\sigma})$ computed directly by QE from the stress tensor are **independent estimates of the same quantity**. Their agreement validates the EOS fit and confirms that the stress tensor is correctly converged.

The third-order BM pressure is:

$$P(V) = \frac{3B_0}{2}\!\left(\eta^7 - \eta^5\right)\!\left[1 + \tfrac{3}{4}(B_0'-4)(\eta^2-1)\right], \quad \eta = \left(\frac{V_0}{V}\right)^{1/3}$$

<details>
<summary>Implementation: <code>bm_pressure_gpa</code> and <code>parabola_pressure_gpa</code> (<code>eos_tools.py</code>)</summary>

```python
def bm_pressure_gpa(V, V0, B0_gpa, B0p):
    eta = (V0 / V) ** (1 / 3)
    return 1.5 * B0_gpa * (eta**7 - eta**5) * (1 + 0.75 * (B0p - 4) * (eta**2 - 1))

def parabola_pressure_gpa(V, poly_v):
    return -poly_v.deriv()(V) * EV_ANG3_TO_GPA
```

</details>

> [!NOTE]
> **Pulay stress.** `pw.x` computes the stress tensor analytically as the derivative of the total energy with respect to cell strain, but it does so *at fixed plane-wave coefficients*. This is an approximation: when the cell deforms, the set of plane waves inside the cutoff sphere should change too, but that change is not included in the analytic formula. The missing contribution is called the **Pulay stress**. It is a systematic bias that makes the calculated pressure look more negative (compressive) than it really is, and it decays only slowly with `ecutwfc` — far more slowly than the total energy, which is protected by the variational principle. The energy-derived pressure from the EOS fit ($-dE/dV$) does **not** suffer from Pulay stress, because each SCF calculation is performed at a fixed cell volume with a self-consistent basis. The residual $\Delta P = P_{\rm QE} - P_{\rm BM}$ in the right-hand plot is therefore a direct measurement of the Pulay stress at the current cutoff.

In [ ]:
from eos_tools import bm_pressure_gpa, parabola_pressure_gpa

P_bm_gpa  = bm_pressure_gpa(V_values, bm['V0'], bm['B0_gpa'], bm['B0p'])
P_par_gpa = parabola_pressure_gpa(V_values, poly_v)

print(f"{'a (Å)':>8}  {'P_QE (GPa)':>11}  {'P_BM (GPa)':>11}  {'P_par (GPa)':>12}  {'dP_BM (GPa)':>12}")
for a, pq, pb, pp in zip(a_values, P_qe_gpa, P_bm_gpa, P_par_gpa):
    print(f'{a:8.4f}  {pq:11.2f}  {pb:11.2f}  {pp:12.2f}  {pq - pb:12.2f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.scatter(a_values, P_qe_gpa, zorder=3, label='QE stress')
ax.plot(a_values, P_bm_gpa,  label='BM fit')
ax.plot(a_values, P_par_gpa, ls='--', label='Parabola fit')
ax.axhline(0, color='gray', lw=0.8, ls=':')
ax.set_xlabel('Lattice parameter $a$ (Å)')
ax.set_ylabel('Pressure (GPa)')
ax.set_title('P vs $a$')
ax.legend()

ax = axes[1]
ax.scatter(a_values, P_qe_gpa - P_bm_gpa, zorder=3, label='$P_{\\rm QE} - P_{\\rm BM}$')
ax.axhline(0, color='gray', lw=0.8, ls=':')
ax.set_xlabel('Lattice parameter $a$ (Å)')
ax.set_ylabel('$\\Delta P$ (GPa)')
ax.set_title('Residual: QE stress − BM fit')
ax.legend()

fig.tight_layout()
plt.show()

### Validation: stress residuals from your chosen cutoff to 85 Ry

The cell below re-runs the EOS sweep from your chosen `ECUTWFC_CONV` up to 85 Ry in steps of 5 Ry and overlays the residuals $\Delta P = P_{\rm QE} - P_{\rm BM}$. Once the cutoff is adequate, the curves should lie close to zero with no systematic slope.

In [ ]:
ECUT_STRESS_VALUES = list(range(ECUTWFC_CONV, 86, 5))

stress_residuals = {}
for ecut in ECUT_STRESS_VALUES:
    cases = [
        (f'eos_ecut{ecut}_{i:02d}', build_mgo_input(
            bulk('MgO', 'rocksalt', a=A_REF * (1 + s)),
            ecut, NK_CONV,
            prefix=f'mgo_eos_ecut{ecut}_{i:02d}',
        ))
        for i, s in enumerate(SCALE_VALUES)
    ]
    results = runner.run_sweep(cases, EOS_DIR, force_rerun=FORCE_RERUN,
                               collect_pressure=True)
    E_ec    = np.array([r['energy_ry'] * RY_TO_EV for r in results])
    P_qe_ec = np.array([r['pressure_kbar'] * 0.1   for r in results])
    bm_ec   = fit_bm_eos(V_values, E_ec)
    P_bm_ec = bm_pressure_gpa(V_values, bm_ec['V0'], bm_ec['B0_gpa'], bm_ec['B0p'])
    stress_residuals[ecut] = P_qe_ec - P_bm_ec

fig, ax = plt.subplots(figsize=(7, 4))
for ecut, residuals in stress_residuals.items():
    ax.plot(a_values, residuals, marker='o', label=f'ecutwfc = {ecut} Ry')
ax.axhline(0, color='gray', lw=0.8, ls=':')
ax.set_xlabel('Lattice parameter $a$ (Å)')
ax.set_ylabel('$\\Delta P = P_{\\rm QE} - P_{\\rm BM}$ (GPa)')
ax.set_title('Pulay stress vs plane-wave cutoff')
ax.legend()
fig.tight_layout()
plt.show()